# **Air Quality Prediction**

In [1]:
import os

import yaml
import joblib
import numpy as np
import pandas as pd

from tqdm import tqdm
from sklearn.model_selection import train_test_split

## 1 - Configuration File

In [2]:
def load_config(config_path):
    """
    Load the configuration file.

    Parameters:
    --------------
    config_path : str
        Configuration file location

    Returns:
    --------------
    params : dict
        Loaded configuration file
    """

    try:
        with open(config_path,'r') as file:
            params = yaml.safe_load(file)
    except FileNotFoundError as err:
        raise RuntimeError(f"Configuration file not found in {config_path}")

    return params

In [3]:
def update_config(key,value,params,path_config):
    """
    Update the configuration file
    """
    # to maintain raw config file immutable
    params = params.copy()
    
    #update the configuration
    params[key] = value

    #write the configuration file
    with open(path_config,'w') as file:
        yaml.dump(params,file)

    print(f"Params updated!! \nKey: {key}  \nValue : {value}")

    # Reload the updated configuration file
    config = load_config(path_config)
    return config
    

In [4]:
PATH_CONFIG = "../config/config.yaml"
config = load_config(PATH_CONFIG)
config

{'columns_datetime': ['tanggal'],
 'columns_object': ['stasiun', 'critical', 'category'],
 'features': ['stasiun', 'pm10', 'pm25', 'so2', 'co', 'o3', 'no2'],
 'columns_int': ['pm10', 'pm25', 'so2', 'co', 'o3', 'no2', 'max'],
 'label': 'category',
 'label_categories': ['BAIK', 'SEDANG', 'TIDAK SEHAT'],
 'label_categories_new': ['BAIK', 'TIDAK BAIK'],
 'path_joined_data': '../data/interim/joined_dataset.pkl',
 'path_raw_data': '../data/raw/',
 'path_validated_data': '../data/interim/validated_data.pkl',
 'range_co': [-1, 47],
 'range_no2': [-1, 65],
 'range_o3': [-1, 151],
 'range_pm10': [-1, 179],
 'range_pm25': [-1, 174],
 'range_so2': [-1, 82],
 'range_stasiun': ['DKI1 (Bunderan HI)',
  'DKI2 (Kelapa Gading)',
  'DKI3 (Jagakarsa)',
  'DKI4 (Lubang Buaya)',
  'DKI5 (Kebon Jeruk) Jakarta Barat']}

## 2 - Data Collection

- Create load_data() function.
- It receives one argument: data_path
- This function load all csv raw data and return the joined dataframe.

In [5]:
def load_data(data_path):
    """
    Load csv files and join into one dataframe.

    Parameters:
    ----------
    path_data : str
        Raw dataset location.

    Returns:
    -------
    raw_dataset : pd.DataFrame
        Loaded and joined data.
    """
    # create dataframe
    raw_dataset = pd.DataFrame()

    for i in tqdm(os.listdir(data_path)) :
        raw_dataset = pd.concat([pd.read_csv(data_path + i),raw_dataset])

    return raw_dataset

    
    
    

In [6]:
PATH_RAW_DATA = config['path_raw_data']
raw_dataset = load_data(PATH_RAW_DATA)

100%|██████████████████████████████████████████████████████████████████████████████████| 12/12 [00:00<00:00, 208.02it/s]


In [7]:
raw_dataset

,tanggal,stasiun,pm10,pm25,so2,co,o3,no2,max,critical,categori
0,2021-09-01,DKI1 (Bunderan HI),63,88,29,15,24,38,88,PM25,SEDANG
1,2021-09-02,DKI1 (Bunderan HI),60,83,29,11,30,28,83,PM25,SEDANG
2,2021-09-03,DKI1 (Bunderan HI),60,82,27,11,37,30,82,PM25,SEDANG
3,2021-09-04,DKI1 (Bunderan HI),58,77,26,10,31,28,77,PM25,SEDANG
4,2021-09-05,DKI1 (Bunderan HI),63,85,27,11,28,28,85,PM25,SEDANG
...,...,...,...,...,...,...,...,...,...,...,...
150,2021-03-27,DKI5 (Kebon Jeruk) Jakarta Barat,51,83,38,12,24,23,83,PM25,SEDANG
151,2021-03-28,DKI5 (Kebon Jeruk) Jakarta Barat,50,78,22,12,34,19,78,PM25,SEDANG
152,2021-03-29,DKI5 (Kebon Jeruk) Jakarta Barat,54,91,21,17,29,26,91,PM25,SEDANG
153,2021-03-30,DKI5 (Kebon Jeruk) Jakarta Barat,44,72,35,8,18,18,72,PM25,SEDANG


In [8]:
raw_dataset = raw_dataset.reset_index(drop=True)

In [9]:
raw_dataset

,tanggal,stasiun,pm10,pm25,so2,co,o3,no2,max,critical,categori
0,2021-09-01,DKI1 (Bunderan HI),63,88,29,15,24,38,88,PM25,SEDANG
1,2021-09-02,DKI1 (Bunderan HI),60,83,29,11,30,28,83,PM25,SEDANG
2,2021-09-03,DKI1 (Bunderan HI),60,82,27,11,37,30,82,PM25,SEDANG
3,2021-09-04,DKI1 (Bunderan HI),58,77,26,10,31,28,77,PM25,SEDANG
4,2021-09-05,DKI1 (Bunderan HI),63,85,27,11,28,28,85,PM25,SEDANG
...,...,...,...,...,...,...,...,...,...,...,...
1825,2021-03-27,DKI5 (Kebon Jeruk) Jakarta Barat,51,83,38,12,24,23,83,PM25,SEDANG
1826,2021-03-28,DKI5 (Kebon Jeruk) Jakarta Barat,50,78,22,12,34,19,78,PM25,SEDANG
1827,2021-03-29,DKI5 (Kebon Jeruk) Jakarta Barat,54,91,21,17,29,26,91,PM25,SEDANG
1828,2021-03-30,DKI5 (Kebon Jeruk) Jakarta Barat,44,72,35,8,18,18,72,PM25,SEDANG


In [10]:
# Serialize the joined dataset.
PATH_JOINED_DATA = f"../data/interim/joined_dataset.pkl"
joblib.dump(raw_dataset, PATH_JOINED_DATA)

['../data/interim/joined_dataset.pkl']

In [11]:
config = update_config(
    key = "path_joined_data",
    value = PATH_JOINED_DATA,
    params = config,
    path_config = PATH_CONFIG
)

Params updated!! 
Key: path_joined_data  
Value : ../data/interim/joined_dataset.pkl


## 3. Data Validation

In [12]:
raw_dataset.dtypes

tanggal        str
stasiun        str
pm10        object
pm25        object
so2         object
co          object
o3          object
no2         object
max         object
critical       str
categori       str
dtype: object

- Several features don't have the same configuration data type.
- We need to handle those error columns.

### 3.1 Handling Column Tanggal

In [13]:
raw_dataset['tanggal'] = pd.to_datetime(raw_dataset['tanggal'])

### 3.2 Handling Column pm10

- ValueError occurs, it tells us that there are data that isn't integer ("---").
- We will replace those "---" with value that don't exists in the column.
- Based on data definition, we know that we can use -1.

In [14]:
# Ensure no single data that is -1.
raw_dataset.eq("-1").any() | raw_dataset.eq(-1).any()

tanggal     False
stasiun     False
pm10        False
pm25        False
so2         False
co          False
o3          False
no2         False
max         False
critical    False
categori    False
dtype: bool

In [15]:
# Replace the "---" with -1 and cast the column into int.
raw_dataset["pm10"] = raw_dataset["pm10"].replace("---", -1).astype(int)

### 3.3 Handling Column pm25

- There is a different ValueError.
- There are NaN values, thus we can't directly convert to int.
- We need to handle this problem first

In [16]:
# Sanity check the missing values.
raw_dataset["pm25"].isna().sum()

np.int64(62)

There are 62 NaN values. For now, we can replace it with -1.

In [17]:
# Replace the NaN values with -1.
raw_dataset["pm25"] = raw_dataset["pm25"].fillna(-1)

# Sanity check the missing values.
raw_dataset["pm25"].isna().sum()

np.int64(0)

In [18]:
# Replace the "---" with -1 and cast the column into int.
raw_dataset["pm25"] = raw_dataset["pm25"].replace("---", -1).astype(int)

### 3.4 Handling Column so2

In [19]:
# Replace the "---" with -1 and cast the column into int.
raw_dataset["so2"] = raw_dataset["so2"].replace("---", -1).astype(int)

### 3.5 Handling Column co

In [20]:
# Replace the "---" with -1 and cast the column into int.
raw_dataset["co"] = raw_dataset["co"].replace("---", -1).astype(int)

### 3.6 Handling Column o3

In [21]:
# Replace the "---" with -1 and cast the column into int.
raw_dataset["o3"] = raw_dataset["o3"].replace("---", -1).astype(int)

### 3.7 Handling Column no2

In [22]:
# Replace the "---" with -1 and cast the column into int.
raw_dataset["no2"] = raw_dataset["no2"].replace("---", -1).astype(int)

### 3.8 Handling Column max

In [23]:
# Try to cast the column to int type.
raw_dataset["max"] = raw_dataset["max"].astype(int)

ValueError: invalid literal for int() with base 10: 'PM25'

- Seems like the error is different.
- There is data with value "PM25" in the max column.
- We meed to investigate the data.

In [24]:
# Check which data that cause error.
raw_dataset[raw_dataset["max"] == "PM25"]

,tanggal,stasiun,pm10,pm25,so2,co,o3,no2,max,critical,categori
1212,2021-12-03,DKI1 (Bunderan HI),49,31,9,19,7,49,PM25,BAIK,NaN


- Looks like there are typos on row index 762.
    - The "BAIK" value must be on categori column.
    - We need to investigate what do max and critical column represents.
- Let's randomly sample 5 data.

In [25]:
raw_dataset.sample(5)

,tanggal,stasiun,pm10,pm25,so2,co,o3,no2,max,critical,categori
715,2021-11-01,DKI5 (Kebon Jeruk) Jakarta Barat,59,91,-1,17,22,27,91,PM25,SEDANG
1053,2021-05-30,DKI5 (Kebon Jeruk) Jakarta Barat,46,66,24,15,18,29,66,PM25,SEDANG
1358,2021-12-25,DKI5 (Kebon Jeruk) Jakarta Barat,55,81,34,17,15,62,81,PM25,SEDANG
267,2021-02-06,DKI5 (Kebon Jeruk) Jakarta Barat,-1,34,21,4,24,6,34,PM25,BAIK
683,2021-11-29,DKI3 (Jagakarsa),29,30,42,7,19,8,42,SO2,BAIK


- Looks like the max column represents the maximum value between any other int columns.
- And looks like the critical column represents the column name of max value.


- Thus, let's fix the error on the row index 762:
    - Replace the max column with pm10 or no2 value, let's take from pm10
    - Replace the critical column with "PM10"
    - Replace the categori column with "BAIK"

In [26]:
# Fix the error.
raw_dataset.loc[1212, "max"] = raw_dataset.loc[762, "pm10"]
raw_dataset.loc[1212, "critical"] = "PM10"
raw_dataset.loc[1212, "categori"] = "BAIK"

In [27]:

# Sanity check the result.
raw_dataset.loc[1212]

tanggal     2021-12-03 00:00:00
stasiun      DKI1 (Bunderan HI)
pm10                         49
pm25                         31
so2                           9
co                           19
o3                            7
no2                          49
max                          40
critical                   PM10
categori                   BAIK
Name: 1212, dtype: object

In [28]:

# Cast the column to int.
raw_dataset["max"] = raw_dataset["max"].astype(int)

### 3.9. Handling Column critical

In [29]:

# Check the unique value.
raw_dataset["critical"].value_counts()

critical
PM25    1631
PM10      65
O3        57
CO        34
SO2       26
Name: count, dtype: int64

- Seems like no action needed.

### 3.10. Handling Column categori

In [30]:
# Check the unique value.
raw_dataset["categori"].value_counts()

categori
SEDANG            1305
TIDAK SEHAT        319
BAIK               189
TIDAK ADA DATA      17
Name: count, dtype: int64

- There are 17 "TIDAK ADA DATA" values, indicate the missing label.
- Since we don't know which label that can replace the "TIDAK ADA DATA", thus we can drop those data.

In [31]:
# Drop the "TIDAK ADA DATA" category.
missing_labels = raw_dataset[raw_dataset["categori"] == "TIDAK ADA DATA"]
raw_dataset = raw_dataset.drop(index = missing_labels.index)

In [32]:
# Sanity check the result.
raw_dataset["categori"].value_counts()

categori
SEDANG         1305
TIDAK SEHAT     319
BAIK            189
Name: count, dtype: int64

- Let's rename the categori column into the proper name, category.
- Don't forget to update the configuration file.

In [33]:

# Rename "categori" into "category".
raw_dataset = raw_dataset.rename(columns = {"categori": "category"})

In [34]:
# Update the configuration parameter.
config = update_config(
    key = "label",
    value = "category",
    params = config,
    path_config = PATH_CONFIG
)

Params updated!! 
Key: label  
Value : category


In [35]:
# Update the configuration parameter.
col_object = config["columns_object"]
col_object[-1] = "category"

config = update_config(
    key = "columns_object",
    value = col_object,
    params = config,
    path_config = PATH_CONFIG
)

Params updated!! 
Key: columns_object  
Value : ['stasiun', 'critical', 'category']


In [36]:

# Sanity check the data types.
raw_dataset.info()

<class 'pandas.DataFrame'>
Index: 1813 entries, 0 to 1829
Data columns (total 11 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   tanggal   1813 non-null   datetime64[us]
 1   stasiun   1813 non-null   str           
 2   pm10      1813 non-null   int64         
 3   pm25      1813 non-null   int64         
 4   so2       1813 non-null   int64         
 5   co        1813 non-null   int64         
 6   o3        1813 non-null   int64         
 7   no2       1813 non-null   int64         
 8   max       1813 non-null   int64         
 9   critical  1813 non-null   str           
 10  category  1813 non-null   str           
dtypes: datetime64[us](1), int64(7), str(3)
memory usage: 170.0 KB


- All columns are already same as in the data definition.
- Now serialized the validated data.

In [37]:
# Serialized the validated data.
PATH_VALIDATED_DATA = f"../data/interim/validated_data.pkl"
joblib.dump(raw_dataset, PATH_VALIDATED_DATA)

['../data/interim/validated_data.pkl']

In [38]:
# Update the configuration parameter.
config = update_config(
    key = "path_validated_data",
    value = PATH_VALIDATED_DATA,
    params = config,
    path_config = PATH_CONFIG
)

Params updated!! 
Key: path_validated_data  
Value : ../data/interim/validated_data.pkl


### 4 - Update the Range of Data in Configuration File

In [39]:
# Update the range of data with the min and max value of each column.
cols = ["pm10", "pm25", "so2", "co", "o3", "no2"]
param_keys = ["range_pm10", "range_pm25", "range_so2", "range_co", "range_o3", "range_no2"]

for col, key in zip(cols, param_keys):
    config = update_config(
        key = key,
        value = [int(np.min(raw_dataset[col])), int(np.max(raw_dataset[col]))],
        params = config,
        path_config = PATH_CONFIG
    )

Params updated!! 
Key: range_pm10  
Value : [-1, 179]
Params updated!! 
Key: range_pm25  
Value : [-1, 174]
Params updated!! 
Key: range_so2  
Value : [-1, 82]
Params updated!! 
Key: range_co  
Value : [-1, 47]
Params updated!! 
Key: range_o3  
Value : [-1, 151]
Params updated!! 
Key: range_no2  
Value : [-1, 65]


In [40]:
# Check the configuration parameters.
config

{'columns_datetime': ['tanggal'],
 'columns_int': ['pm10', 'pm25', 'so2', 'co', 'o3', 'no2', 'max'],
 'columns_object': ['stasiun', 'critical', 'category'],
 'features': ['stasiun', 'pm10', 'pm25', 'so2', 'co', 'o3', 'no2'],
 'label': 'category',
 'label_categories': ['BAIK', 'SEDANG', 'TIDAK SEHAT'],
 'label_categories_new': ['BAIK', 'TIDAK BAIK'],
 'path_joined_data': '../data/interim/joined_dataset.pkl',
 'path_raw_data': '../data/raw/',
 'path_validated_data': '../data/interim/validated_data.pkl',
 'range_co': [-1, 47],
 'range_no2': [-1, 65],
 'range_o3': [-1, 151],
 'range_pm10': [-1, 179],
 'range_pm25': [-1, 174],
 'range_so2': [-1, 82],
 'range_stasiun': ['DKI1 (Bunderan HI)',
  'DKI2 (Kelapa Gading)',
  'DKI3 (Jagakarsa)',
  'DKI4 (Lubang Buaya)',
  'DKI5 (Kebon Jeruk) Jakarta Barat']}

## 5 - Data Defense

- Create the check_data() function.
- It receives 2 arguments: input_data and params
    - input_data is the raw dataset
    - params is the configuration parameters
- It is a void function (no return value).
- If AssertionError happens, there are exists data that don't match the configuration.

In [41]:
# Function to do data defense.
def check_data(input_data, params):
    """
    Do data defense for checking the data types and range of data.

    Parameters:
    ----------
    input_data : pd.DataFrame
        The data to be checked.

    params : dict
        Loaded configuration parameters.

    Returns:
    -------
    None, it's a void function.
    """

    # Check data types.
    assert input_data.select_dtypes("datetime").columns.to_list() == params["columns_datetime"], "an error occurs in datetime column(s)."
    assert input_data.select_dtypes("str").columns.to_list() == params["columns_object"], "an error occurs in object column(s)."
    assert input_data.select_dtypes("number").columns.to_list() == params["columns_int"], "an error occurs in int32 column(s)."

    # Check range of data.
    assert set(input_data['stasiun']).issubset(set(params['range_stasiun'])), "an error occurs in stasiun range."
    assert input_data['pm10'].between(params['range_pm10'][0], params['range_pm10'][1]).sum() == len(input_data), "an error occurs in pm10 range."
    assert input_data['pm25'].between(params['range_pm25'][0], params['range_pm25'][1]).sum() == len(input_data), "an error occurs in pm25 range."
    assert input_data['so2'].between(params['range_so2'][0], params['range_so2'][1]).sum() == len(input_data), "an error occurs in so2 range."
    assert input_data['co'].between(params['range_co'][0], params['range_co'][1]).sum() == len(input_data), "an error occurs in co range."
    assert input_data['o3'].between(params['range_o3'][0], params['range_o3'][1]).sum() == len(input_data), "an error occurs in o3 range."
    assert input_data['no2'].between(params['range_no2'][0], params['range_no2'][1]).sum() == len(input_data), "an error occurs in no2 range."

In [42]:
# Do data defense.
check_data(raw_dataset, config)

Seems like our data are in good condition!

## 6 - Data Split

In [44]:
# Input-Output Split.
def split_input_output(input_data, params):
    """
    Split the input(X) and output (y).

    Parameters:
    ----------
    input_data : pd.DataFrame
        The processed dataset.

    params : dict
        Loaded configuration parameters.

    Returns:
    -------
    X : pd.DataFrame
        The input data.

    y : pd.Series
        The output data.
    """

    X = input_data[params["features"]].copy()
    y = input_data[params["label"]].copy()

    print(f"Original data shape : {input_data.shape}")
    print(f"Selected Features   : {params["features"]}")
    print(f"X data shape        : {X.shape}")
    print(f"y data shape        : {y.shape}")

    return X, y

In [45]:
X, y = split_input_output(
    input_data = raw_dataset,
    params = config
)

Original data shape : (1813, 11)
Selected Features   : ['stasiun', 'pm10', 'pm25', 'so2', 'co', 'o3', 'no2']
X data shape        : (1813, 7)
y data shape        : (1813,)


In [46]:
# Sanity check the input (X).
X.head()

,stasiun,pm10,pm25,so2,co,o3,no2
0,DKI1 (Bunderan HI),63,88,29,15,24,38
1,DKI1 (Bunderan HI),60,83,29,11,30,28
2,DKI1 (Bunderan HI),60,82,27,11,37,30
3,DKI1 (Bunderan HI),58,77,26,10,31,28
4,DKI1 (Bunderan HI),63,85,27,11,28,28


In [47]:
# Sanity check the output (y).
y.head()

0    SEDANG
1    SEDANG
2    SEDANG
3    SEDANG
4    SEDANG
Name: category, dtype: str

In [48]:
# Train-Test Split.
def split_train_test(X, y, test_size, random_state):
    """
    Split the train and test set.

    Parameters:
    ----------
    X : pd.DataFrame
        The input data.

    y : pd.Series
        The output data.

    test_size : float
        The proportion of test set.

    random_state : int
        For reproducibility

    Returns:
    -------
    X_train, X_test : pd.DataFrame
        The train and test input.

    y_train, y_test : pd.Series
        The train and test output.
    """

    X_train, X_test, y_train, y_test = train_test_split(
                                            X, y,
                                            test_size = test_size,
                                            random_state = random_state,
                                            stratify = y
                                       )

    print(f"X_train shape : {X_train.shape}")
    print(f"y_train shape : {y_train.shape}")
    print(f"X_test shape  : {X_test.shape}")
    print(f"y_test shape  : {y_test.shape}")

    return X_train, X_test, y_train, y_test

In [49]:
# train:valid:test -> 80:10:10

RANDOM_STATE = 123

# Train vs not-train.
X_train, X_not_train, y_train, y_not_train = split_train_test(
    X = X,
    y = y,
    test_size = 0.2,
    random_state = RANDOM_STATE
)

print()
# Valid vs test.
X_valid, X_test, y_valid, y_test = split_train_test(
    X = X_not_train,
    y = y_not_train,
    test_size = 0.5,
    random_state = RANDOM_STATE
)

X_train shape : (1450, 7)
y_train shape : (1450,)
X_test shape  : (363, 7)
y_test shape  : (363,)

X_train shape : (181, 7)
y_train shape : (181,)
X_test shape  : (182, 7)
y_test shape  : (182,)


In [50]:
# Serialize the splitted data.
PATH_SPLITTED_DATA = f"../data/interim/"

joblib.dump(X_train, f"{PATH_SPLITTED_DATA}X_train.pkl")
joblib.dump(y_train, f"{PATH_SPLITTED_DATA}y_train.pkl")
joblib.dump(X_valid, f"{PATH_SPLITTED_DATA}X_valid.pkl")
joblib.dump(y_valid, f"{PATH_SPLITTED_DATA}y_valid.pkl")
joblib.dump(X_test, f"{PATH_SPLITTED_DATA}X_test.pkl")
joblib.dump(y_test, f"{PATH_SPLITTED_DATA}y_test.pkl")

['../data/interim/y_test.pkl']

In [51]:
# Update the configuration parameters.
config = update_config(
    key = "path_train_set",
    value = [f"{PATH_SPLITTED_DATA}X_train.pkl", f"{PATH_SPLITTED_DATA}y_train.pkl"],
    params = config,
    path_config = PATH_CONFIG
)

config = update_config(
    key = "path_valid_set",
    value = [f"{PATH_SPLITTED_DATA}X_valid.pkl", f"{PATH_SPLITTED_DATA}y_valid.pkl"],
    params = config,
    path_config = PATH_CONFIG
)

config = update_config(
    key = "path_test_set",
    value = [f"{PATH_SPLITTED_DATA}X_test.pkl", f"{PATH_SPLITTED_DATA}y_test.pkl"],
    params = config,
    path_config = PATH_CONFIG
)

Params updated!! 
Key: path_train_set  
Value : ['../data/interim/X_train.pkl', '../data/interim/y_train.pkl']
Params updated!! 
Key: path_valid_set  
Value : ['../data/interim/X_valid.pkl', '../data/interim/y_valid.pkl']
Params updated!! 
Key: path_test_set  
Value : ['../data/interim/X_test.pkl', '../data/interim/y_test.pkl']


In [52]:
# Check the configuration parameters.
config

{'columns_datetime': ['tanggal'],
 'columns_int': ['pm10', 'pm25', 'so2', 'co', 'o3', 'no2', 'max'],
 'columns_object': ['stasiun', 'critical', 'category'],
 'features': ['stasiun', 'pm10', 'pm25', 'so2', 'co', 'o3', 'no2'],
 'label': 'category',
 'label_categories': ['BAIK', 'SEDANG', 'TIDAK SEHAT'],
 'label_categories_new': ['BAIK', 'TIDAK BAIK'],
 'path_joined_data': '../data/interim/joined_dataset.pkl',
 'path_raw_data': '../data/raw/',
 'path_test_set': ['../data/interim/X_test.pkl', '../data/interim/y_test.pkl'],
 'path_train_set': ['../data/interim/X_train.pkl',
  '../data/interim/y_train.pkl'],
 'path_valid_set': ['../data/interim/X_valid.pkl',
  '../data/interim/y_valid.pkl'],
 'path_validated_data': '../data/interim/validated_data.pkl',
 'range_co': [-1, 47],
 'range_no2': [-1, 65],
 'range_o3': [-1, 151],
 'range_pm10': [-1, 179],
 'range_pm25': [-1, 174],
 'range_so2': [-1, 82],
 'range_stasiun': ['DKI1 (Bunderan HI)',
  'DKI2 (Kelapa Gading)',
  'DKI3 (Jagakarsa)',
  'DKI4